# A2 — GBDT Re-Ranker Pipeline (Q1–Q5, Q9)
## CS4.406 — Information Retrieval & Extraction

**Track: GBDT re-ranker (Q2 Option A).** The team's other option (NRMS, Q2 Option B) is a separate notebook; this one is the GBDT side, built to the same scope for a side-by-side comparison before deciding what goes into the report/resubmission.

Unlike a from-scratch notebook, every cell below imports and runs this repo's actual `src/ire_a1`/`src/ire_a2` package and `scripts/` code -- tested, reproducible via `make`, and the same code a script run from the terminal would execute. This notebook is a presentation layer over that code, not a second implementation of it.

| Section | What | Source |
|---|---|---|
| 0 | Setup | -- |
| 1 | Q1 — Feature engineering | `src/ire_a2/features.py` |
| 2 | Three baselines (B1 BM25, B2 Embeddings, B3 Hybrid) + Q2 GBDT + Q3 ablation | `scripts/train_reranker.py` |
| 3 | Q4 — Serving & scale analysis | `scripts/reranker_scale_analysis.py` |
| 4 | Q5 — Extended evaluation (beyond-accuracy, 2 slices) | `scripts/run_reranker_extended_eval.py` |
| 5 | Q9 — Anti-leakage | `tests/test_a2_features.py`, `scripts/run_reranker_leakage_ablation.py` |
| 6 | Closing note | -- |


## 0. Setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "scripts"))

DATASETS = [("ebnerd", "demo"), ("mind", "small")]

for dataset, scale in DATASETS:
    d = REPO_ROOT / "data" / "processed" / dataset / scale
    assert (d / "reranker_features_train.parquet").exists(), (
        f"{d}/reranker_features_train.parquet missing -- run "
        f"`make reranker-features DATASET={dataset} SCALE={scale}` first"
    )
print("Data confirmed built for:", DATASETS)


Data confirmed built for: [('ebnerd', 'demo'), ('mind', 'small')]


## 1. Q1 — Feature Engineering

`ire_a2.features.FeatureBuilder` computes every candidate-level feature from `UserHistoryIndex.recent()` (strictly-before-cutoff click history) or a train-only frozen popularity dict -- see `src/ire_a2/features.py`'s module docstring for the full behaviour-window argument. `scripts/build_reranker_features.py` (Q1's CLI) already ran this for both datasets; this cell just peeks at the cached output.


In [2]:
import polars as pl
from ire_a2.features import FEATURE_COLUMNS

print("Feature columns (Q1):")
for c in FEATURE_COLUMNS:
    print(" ", c)

for dataset, scale in DATASETS:
    d = REPO_ROOT / "data" / "processed" / dataset / scale
    frame = pl.read_parquet(d / "reranker_features_train.parquet")
    print(f"\n{dataset}/{scale}: {frame.height:,} rows, {frame['impression_id'].n_unique():,} impressions")
    print(frame.select(FEATURE_COLUMNS[:6]).describe())


Feature columns (Q1):
  bm25_score
  embedding_score
  candidate_position
  hist_click_count
  hist_recency_weighted_count
  hist_category_match
  hist_category_match_weighted
  candidate_popularity
  freshness_hours
  has_freshness
  hist_avg_dwell_seconds
  session_ordinal
  session_click_count_so_far

ebnerd/demo: 278,139 rows, 24,724 impressions
shape: (9, 7)
┌────────────┬────────────┬──────────────┬──────────────┬──────────────┬─────────────┬─────────────┐
│ statistic  ┆ bm25_score ┆ embedding_sc ┆ candidate_po ┆ hist_click_c ┆ hist_recenc ┆ hist_catego │
│ ---        ┆ ---        ┆ ore          ┆ sition       ┆ ount         ┆ y_weighted_ ┆ ry_match    │
│ str        ┆ f64        ┆ ---          ┆ ---          ┆ ---          ┆ count       ┆ ---         │
│            ┆            ┆ f64          ┆ f64          ┆ f64          ┆ ---         ┆ f64         │
│            ┆            ┆              ┆              ┆              ┆ f64         ┆             │
╞════════════╪════════════╪═


mind/small: 5,723,002 rows, 153,727 impressions


shape: (9, 7)
┌────────────┬────────────┬──────────────┬──────────────┬──────────────┬─────────────┬─────────────┐
│ statistic  ┆ bm25_score ┆ embedding_sc ┆ candidate_po ┆ hist_click_c ┆ hist_recenc ┆ hist_catego │
│ ---        ┆ ---        ┆ ore          ┆ sition       ┆ ount         ┆ y_weighted_ ┆ ry_match    │
│ str        ┆ f64        ┆ ---          ┆ ---          ┆ ---          ┆ count       ┆ ---         │
│            ┆            ┆ f64          ┆ f64          ┆ f64          ┆ ---         ┆ f64         │
│            ┆            ┆              ┆              ┆              ┆ f64         ┆             │
╞════════════╪════════════╪══════════════╪══════════════╪══════════════╪═════════════╪═════════════╡
│ count      ┆ 5.723002e6 ┆ 5.723002e6   ┆ 5.723002e6   ┆ 5.723002e6   ┆ 5.723002e6  ┆ 5.723002e6  │
│ null_count ┆ 0.0        ┆ 0.0          ┆ 0.0          ┆ 0.0          ┆ 0.0         ┆ 0.0         │
│ mean       ┆ 12.435846  ┆ 0.154999     ┆ 38.099419    ┆ 15.063116    ┆ 7.24

## 2. Three Baselines (B1/B2/B3) + Q2 GBDT Re-Ranker + Q3 Ablation

`scripts/train_reranker.py` does all three of these in one pass per dataset: evaluates B1 (BM25 alone), B2 (embeddings alone), and B3 (a learned-weight BM25+embedding hybrid fit on a *train*-split sample -- see `ire_a2/baselines.py`), trains the GBDT re-ranker (Q2 Option A) and evaluates it on the same val sample, then reports Q3's paired-bootstrap-95%-CI ablation of GBDT vs. each baseline (a claimed gain must exclude zero to count as significant).


In [3]:
from train_reranker import run as run_reranker

reranker_csv_rows, reranker_ablation_rows = [], []
for dataset, scale in DATASETS:
    run_reranker(dataset, scale, reranker_csv_rows, reranker_ablation_rows)



=== ebnerd/demo ===


training on 278,139 rows (24,724 impressions)


B3 hybrid: learned alpha=0.00 on 5,000 train-sample impressions


evaluating on 5,000 sampled impressions (seed=42, same as Q4)



-- bm25 --
    metric |     mean |            95% CI |      n
       AUC |   0.4944 | [0.4850, 0.5037] |  5,000
       MRR |   0.3122 | [0.3046, 0.3207] |  5,000
    nDCG@5 |   0.3413 | [0.3321, 0.3504] |  5,000
   nDCG@10 |   0.4270 | [0.4194, 0.4344] |  5,000

-- embeddings --
    metric |     mean |            95% CI |      n
       AUC |   0.5391 | [0.5308, 0.5480] |  5,000
       MRR |   0.3384 | [0.3303, 0.3459] |  5,000
    nDCG@5 |   0.3751 | [0.3654, 0.3842] |  5,000
   nDCG@10 |   0.4536 | [0.4457, 0.4612] |  5,000

-- hybrid --
    metric |     mean |            95% CI |      n
       AUC |   0.5391 | [0.5308, 0.5480] |  5,000
       MRR |   0.3384 | [0.3303, 0.3459] |  5,000
    nDCG@5 |   0.3751 | [0.3654, 0.3842] |  5,000
   nDCG@10 |   0.4536 | [0.4457, 0.4612] |  5,000


...

B3 hybrid: learned alpha=0.00 on 5,000 train-sample impressions


evaluating on 5,000 sampled impressions (seed=42, same as Q4)



-- bm25 --
    metric |     mean |            95% CI |      n
       AUC |   0.5529 | [0.5443, 0.5612] |  5,000
       MRR |   0.2934 | [0.2849, 0.3025] |  5,000
    nDCG@5 |   0.2688 | [0.2592, 0.2786] |  5,000
   nDCG@10 |   0.3308 | [0.3224, 0.3398] |  5,000

-- embeddings --
    metric |     mean |            95% CI |      n
       AUC |   0.6333 | [0.6251, 0.6416] |  5,000
       MRR |   0.3402 | [0.3315, 0.3492] |  5,000
    nDCG@5 |   0.3247 | [0.3151, 0.3345] |  5,000
   nDCG@10 |   0.3843 | [0.3755, 0.3932] |  5,000

-- hybrid --
    metric |     mean |            95% CI |      n
       AUC |   0.6333 | [0.6251, 0.6416] |  5,000
       MRR |   0.3402 | [0.3315, 0.3492] |  5,000
    nDCG@5 |   0.3247 | [0.3151, 0.3345] |  5,000
   nDCG@10 |   0.3843 | [0.3755, 0.3932] |  5,000



-- gbdt_reranker --
    metric |     mean |            95% CI |      n
       AUC |   0.5937 | [0.5847, 0.6020] |  5,000
       MRR |   0.3395 | [0.3303, 0.3493] |  5,000
    nDCG@5 |   0.3181 | [0.3086, 0.3281] |  5,000
   nDCG@10 |   0.3733 | [0.3641, 0.3828] |  5,000

top feature importances (gain):
          candidate_popularity   488707.9
               embedding_score   278464.0
                    bm25_score    66444.5
   hist_recency_weighted_count    60074.9
           hist_category_match    56264.7
            candidate_position    44654.6
  hist_category_match_weighted    30026.3
              hist_click_count    23715.7

-- GBDT vs bm25 (paired bootstrap ablation) --
         AUC: delta=+0.0409 [+0.0293, +0.0517]  SIGNIFICANT
         MRR: delta=+0.0461 [+0.0368, +0.0559]  SIGNIFICANT
      nDCG@5: delta=+0.0493 [+0.0399, +0.0595]  SIGNIFICANT
     nDCG@10: delta=+0.0426 [+0.0344, +0.0510]  SIGNIFICANT

-- GBDT vs embeddings (paired bootstrap ablation) --
         AUC: del

         MRR: delta=-0.0007 [-0.0093, +0.0087]  not significant
      nDCG@5: delta=-0.0066 [-0.0158, +0.0025]  not significant
     nDCG@10: delta=-0.0110 [-0.0189, -0.0029]  SIGNIFICANT


## 3. Q4 — Serving & Scale Analysis

`scripts/reranker_scale_analysis.py`: index/feature-store memory footprint, p99 single-request latency (candidate generation via embedding search top-K=200 → Q1 feature build → GBDT score), and a back-of-envelope cost per 1,000 queries at a 100ms p99 SLA.


In [4]:
from reranker_scale_analysis import run as run_scale_analysis

scale_csv_rows = []
for dataset, scale in DATASETS:
    run_scale_analysis(dataset, scale, scale_csv_rows)



=== ebnerd/demo ===


memory: BM25 index 5.9 MB | embedding matrix 18.1 MB | feature store 0.3 MB
timing 200 single-request runs (candidate gen -> feature build -> GBDT score)



           stage |  p50 (ms) |  p99 (ms)
       retrieval |      0.61 |      0.97
   feature build |      1.77 |      2.03
      gbdt score |      0.90 |      1.02
           TOTAL |      3.30 |      3.85

single-core capacity: 259.6 req/s (at measured p99 latency)
meets 100ms p99 SLA on a single core: True
back-of-envelope cost per 1,000 queries: $0.0001 (@ $0.05/vCPU-hour, single core, stated assumption)



=== mind/small ===


memory: BM25 index 42.5 MB | embedding matrix 100.2 MB | feature store 1.2 MB


timing 200 single-request runs (candidate gen -> feature build -> GBDT score)



           stage |  p50 (ms) |  p99 (ms)
       retrieval |      2.93 |      3.63
   feature build |      2.68 |      3.35
      gbdt score |      1.06 |      1.18
           TOTAL |      6.67 |      7.54

single-core capacity: 132.7 req/s (at measured p99 latency)
meets 100ms p99 SLA on a single core: True
back-of-envelope cost per 1,000 queries: $0.0001 (@ $0.05/vCPU-hour, single core, stated assumption)


### Scaling argument: what breaks first at 10x

- **Catalog / candidate-generation.** Embedding retrieval here is brute-force cosine similarity over the full catalog (`EmbeddingIndex.search()`), O(catalog size) per query. At 10x articles that's a 10x slowdown on the single largest chunk of per-request latency measured above -- the first thing to swap for an ANN index (FAISS/ScaNN), exactly as `embeddings.py`'s own docstring already flags.
- **Feature building.** `FeatureBuilder.build_rows()` and `EbnerdSessionIndex`'s per-user session scan are plain Python loops over each candidate/history item. At 10x candidates-per-request (K=200 → 2000) this stage's measured latency scales roughly linearly and becomes the new bottleneck once candidate generation is ANN-accelerated -- next to vectorize (polars/numpy over the whole candidate batch) or move to a feature store with precomputed per-article/per-user aggregates refreshed on a schedule, rather than recomputed per request.
- **GBDT scoring.** Cheapest stage today (single-digit ms) and stays cheap at 10x -- LightGBM's `predict()` is already vectorized over the candidate batch; the real cost at 10x is periodic retraining on a much larger labeled set, which is an offline batch job, not on the request path.
- **Memory.** BM25 postings + embedding matrix scale roughly linearly with catalog size; at 10x MIND's article count that's still tens/low hundreds of MB (comfortably in-memory on one machine per the measurement above) -- not the first thing to break.
- **Net:** candidate generation (full-corpus brute-force search) breaks first, well before feature building or scoring; an ANN index is the first infrastructure change this system would need at meaningfully higher scale.


## 4. Q5 — Extended Evaluation

`scripts/run_reranker_extended_eval.py`: the full metric set (AUC, MRR, nDCG@5, nDCG@10, diversity, novelty, coverage) for all four methods, sliced two ways (cold-start vs. warm on history length; head vs. tail on the clicked article's train-split popularity), bootstrap 95% CIs throughout -- reusing `run_eval_harness.py`'s own `attach_beyond_accuracy()`/`aggregate()` unmodified.


In [5]:
from run_reranker_extended_eval import run as run_extended_eval

extended_csv_rows = []
for dataset, scale in DATASETS:
    run_extended_eval(dataset, scale, extended_csv_rows)



=== ebnerd/demo ===


popularity: 1,114 distinct clicked articles, head/tail threshold=34 clicks


evaluating on 5,000 sampled impressions (B3 alpha=0.00)



-- bm25: overall --
      metric |     mean |            95% CI |      n
         AUC |   0.4944 | [0.4850, 0.5037] |  5,000
         MRR |   0.3122 | [0.3046, 0.3207] |  5,000
      nDCG@5 |   0.3413 | [0.3321, 0.3504] |  5,000
     nDCG@10 |   0.4270 | [0.4194, 0.4344] |  5,000
 Diversity@5 |   0.8421 | [0.8402, 0.8440] |  5,000
   Novelty@5 |  14.7472 | [14.7255, 14.7685] |  5,000
  Coverage@5 |   0.1411 | [0.1245, 0.1289] |  5,000

-- bm25: cold_start (n=0) --
      metric |     mean |            95% CI |      n
         AUC |      nan | [nan, nan] |      0
         MRR |      nan | [nan, nan] |      0
      nDCG@5 |      nan | [nan, nan] |      0
     nDCG@10 |      nan | [nan, nan] |      0
 Diversity@5 |      nan | [nan, nan] |      0
   Novelty@5 |      nan | [nan, nan] |      0
  Coverage@5 |      nan | [nan, nan] |      0



-- bm25: warm (n=5,000) --
      metric |     mean |            95% CI |      n
         AUC |   0.4944 | [0.4850, 0.5037] |  5,000
         MRR |   0.3122 | [0.3046, 0.3207] |  5,000
      nDCG@5 |   0.3413 | [0.3321, 0.3504] |  5,000
     nDCG@10 |   0.4270 | [0.4194, 0.4344] |  5,000
 Diversity@5 |   0.8421 | [0.8402, 0.8440] |  5,000
   Novelty@5 |  14.7472 | [14.7255, 14.7685] |  5,000
  Coverage@5 |   0.1411 | [0.1245, 0.1289] |  5,000


...


-- hybrid: head (n=1,433) --
      metric |     mean |            95% CI |      n
         AUC |   0.6597 | [0.6444, 0.6757] |  1,433
         MRR |   0.3975 | [0.3783, 0.4147] |  1,433
      nDCG@5 |   0.3878 | [0.3668, 0.4068] |  1,433
     nDCG@10 |   0.4361 | [0.4165, 0.4527] |  1,433
 Diversity@5 |   0.8545 | [0.8490, 0.8599] |  1,433
   Novelty@5 |  15.0280 | [14.9525, 15.0904] |  1,433
  Coverage@5 |   0.0101 | [0.0080, 0.0086] |  1,433



-- gbdt_reranker: overall --
      metric |     mean |            95% CI |      n
         AUC |   0.5937 | [0.5847, 0.6020] |  5,000
         MRR |   0.3395 | [0.3303, 0.3493] |  5,000
      nDCG@5 |   0.3181 | [0.3086, 0.3281] |  5,000
     nDCG@10 |   0.3733 | [0.3641, 0.3828] |  5,000
 Diversity@5 |   0.8942 | [0.8923, 0.8960] |  5,000
   Novelty@5 |  14.9377 | [14.8887, 14.9859] |  5,000
  Coverage@5 |   0.0118 | [0.0097, 0.0103] |  5,000

-- gbdt_reranker: cold_start (n=558) --
      metric |     mean |            95% CI |      n
         AUC |   0.5930 | [0.5636, 0.6205] |    558
         MRR |   0.3661 | [0.3346, 0.3951] |    558
      nDCG@5 |   0.3612 | [0.3282, 0.3930] |    558
     nDCG@10 |   0.4109 | [0.3802, 0.4388] |    558
 Diversity@5 |   0.9080 | [0.9026, 0.9136] |    558
   Novelty@5 |  15.1406 | [14.9734, 15.2812] |    558
  Coverage@5 |   0.0048 | [0.0036, 0.0041] |    558



-- gbdt_reranker: warm (n=4,442) --
      metric |     mean |            95% CI |      n
         AUC |   0.5938 | [0.5849, 0.6029] |  4,442
         MRR |   0.3362 | [0.3256, 0.3474] |  4,442
      nDCG@5 |   0.3127 | [0.3023, 0.3244] |  4,442
     nDCG@10 |   0.3686 | [0.3590, 0.3790] |  4,442
 Diversity@5 |   0.8924 | [0.8904, 0.8944] |  4,442
   Novelty@5 |  14.9123 | [14.8597, 14.9660] |  4,442
  Coverage@5 |   0.0112 | [0.0092, 0.0098] |  4,442



-- gbdt_reranker: tail (n=3,567) --
      metric |     mean |            95% CI |      n
         AUC |   0.5352 | [0.5258, 0.5458] |  3,567
         MRR |   0.2970 | [0.2857, 0.3085] |  3,567
      nDCG@5 |   0.2661 | [0.2548, 0.2778] |  3,567
     nDCG@10 |   0.3250 | [0.3144, 0.3361] |  3,567
 Diversity@5 |   0.8926 | [0.8904, 0.8946] |  3,567
   Novelty@5 |  15.3208 | [15.2628, 15.3812] |  3,567
  Coverage@5 |   0.0105 | [0.0084, 0.0090] |  3,567



-- gbdt_reranker: head (n=1,433) --
      metric |     mean |            95% CI |      n
         AUC |   0.7394 | [0.7251, 0.7516] |  1,433
         MRR |   0.4452 | [0.4262, 0.4630] |  1,433
      nDCG@5 |   0.4477 | [0.4290, 0.4660] |  1,433
     nDCG@10 |   0.4935 | [0.4757, 0.5104] |  1,433
 Diversity@5 |   0.8981 | [0.8942, 0.9020] |  1,433
   Novelty@5 |  13.9843 | [13.9117, 14.0567] |  1,433
  Coverage@5 |   0.0068 | [0.0055, 0.0060] |  1,433


## 5. Q9 — Anti-Leakage

**Boundary-enforcement test** (Q9's "include a test asserting this"): `tests/test_a2_features.py::test_user_history_index_recent_excludes_clicks_at_or_after_cutoff` -- re-run live below, not just referenced, using the exact same toy setup as the test file.


In [6]:
from datetime import datetime

import polars as pl

from ire_a1.feature_store import UserHistoryIndex

user_history = pl.DataFrame({
    "user_id": ["u1", "u1", "u1"],
    "dataset": ["mind"] * 3,
    "article_id": ["A1", "A2", "A3"],
    "click_timestamp": [datetime(2024, 1, 1, 10), datetime(2024, 1, 2, 10), datetime(2024, 1, 3, 10)],
})
articles = pl.DataFrame({"article_id": ["A1", "A2", "A3", "A4"], "title": ["t1", "t2", "t3", "t4"]})
idx = UserHistoryIndex(user_history, articles)
cutoff = datetime(2024, 1, 3, 10)  # A3 clicks exactly at this cutoff -- must be excluded

recent = idx.recent("u1", cutoff, max_n=20)
seen_ids = [aid for _ts, aid, _title in recent]
assert "A3" not in seen_ids and seen_ids == ["A2", "A1"], seen_ids
print("Boundary check PASSED: click at/after cutoff excluded, remaining history most-recent-first ->", seen_ids)


Boundary check PASSED: click at/after cutoff excluded, remaining history most-recent-first -> ['A2', 'A1']


**With/without-leak ablation:** `scripts/run_reranker_leakage_ablation.py` deliberately trains a second GBDT whose `candidate_popularity` feature is computed with train **+ val** hindsight click counts (via `FeatureBuilder`'s `popularity_override` hook) instead of the safe train-only default, then reports a paired bootstrap 95% CI on the metric deltas.


In [7]:
from run_reranker_leakage_ablation import run as run_leakage_ablation

leakage_csv_rows = []
for dataset, scale in DATASETS:
    run_leakage_ablation(dataset, scale, leakage_csv_rows)



=== ebnerd/demo ===


train sample: 2,000 impressions | eval sample: 5,000 impressions


  safe: trained on 23,049 rows (2,000 impressions)


  leaky: trained on 23,049 rows (2,000 impressions)



    metric |  without leak |     with leak |     delta | significant
       AUC |        0.5442 |        0.6933 |   +0.1490 | YES


...

train sample: 2,000 impressions | eval sample: 5,000 impressions


  safe: trained on 70,950 rows (2,000 impressions)


  leaky: trained on 70,950 rows (2,000 impressions)



    metric |  without leak |     with leak |     delta | significant
       AUC |        0.5685 |        0.6364 |   +0.0679 | YES


       MRR |        0.2969 |        0.3446 |   +0.0478 | YES
    nDCG@5 |        0.2768 |        0.3237 |   +0.0469 | YES
   nDCG@10 |        0.3348 |        0.3840 |   +0.0492 | YES


## 6. Closing Note

**Q6 (Codabench resubmission) is deliberately not run here.** Both re-ranker tracks (this GBDT notebook and the teammate's NRMS notebook) now have complete, comparable Q1–Q5/Q9 results on `demo`/`small` scale. Which track's predictions actually get regenerated at large scale and resubmitted to Codabench is a decision for after comparing both notebooks' results side by side -- not before.
